In [2]:
"""Monthly qPCR vs. sequencing positivity rate, per virus (Influenza A, Influenza
B, RSV A, RSV B, SARS-CoV-2 -- each plotted separately, not A+B combined).
Two SEPARATE figures per virus: a time series of both monthly rates, and a
scatter/correlation of the two rates against each other. (x-axis of the time
series panel is sized to match plot_first_detection_dumbbell plots.)

Monthly rate = (# detected) / (# attempted, i.e. all kept rows) that month,
grouped by qPCR collection month (qpcr_date), separately for qpcr_detected
and sequencing_detected.
"""

#Import required packages
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from scipy.stats import pearsonr, spearmanr

DATA_PATH = "Data_3_Correlation_Data_Final.xlsx"

COLOR_QPCR = "#2a78d6"   #  (blue)
COLOR_SEQ = "#eb6834"    #  (orange)

# Must match plot_first_detection_dumbbell_nodallas.py exactly: same total
# figure width + same left/right subplot-margin fractions -> the Oct
# 2022-Jul 2026 axis renders at the identical physical/pixel width in both
# PNGs, so the two chart families line up if stacked or viewed side by side.
FIG_WIDTH_IN = 14
AXES_LEFT = 0.07
AXES_RIGHT = 0.97

# All 5 viruses' source sheets share the same raw date range (qpcr_date /
# metadata_date both run 2022-12-05/06 -> 2026-07-15)
SHARED_XLIM = (pd.Timestamp("2022-10-01"), pd.Timestamp("2026-08-01"))

VIRUSES = [
    {
        "sheet": "fluA",
        "title": "Influenza A",
        "out_path_ts": "monthly_qpcr_vs_sequencing_positivity_fluA_timeseries.png",
        "out_path_scatter": "monthly_qpcr_vs_sequencing_positivity_fluA_correlation.png",
    },
    {
        "sheet": "fluB",
        "title": "Influenza B",
        "out_path_ts": "monthly_qpcr_vs_sequencing_positivity_fluB_timeseries.png",
        "out_path_scatter": "monthly_qpcr_vs_sequencing_positivity_fluB_correlation.png",
    },
    {
        "sheet": "rsvA",
        "title": "RSV A",
        "out_path_ts": "monthly_qpcr_vs_sequencing_positivity_rsvA_timeseries.png",
        "out_path_scatter": "monthly_qpcr_vs_sequencing_positivity_rsvA_correlation.png",
    },
    {
        "sheet": "rsvB",
        "title": "RSV B",
        "out_path_ts": "monthly_qpcr_vs_sequencing_positivity_rsvB_timeseries.png",
        "out_path_scatter": "monthly_qpcr_vs_sequencing_positivity_rsvB_correlation.png",
    },
    {
        "sheet": "sars-cov-2",
        "title": "SARS-CoV-2",
        "out_path_ts": "monthly_qpcr_vs_sequencing_positivity_sarscov2_timeseries.png",
        "out_path_scatter": "monthly_qpcr_vs_sequencing_positivity_sarscov2_correlation.png",
    },
]


def load_monthly_rates(sheet):
    df = pd.read_excel(DATA_PATH, sheet_name=sheet)
    before = len(df)
    df = df[df["mismatch_flag_after_correction"].astype(str).str.lower() == "none"].copy()
    df["qpcr_date"] = pd.to_datetime(df["qpcr_date"])
    df["month"] = df["qpcr_date"].dt.to_period("M")

    monthly = df.groupby("month").agg(
        n=("qpcr_detected", "size"),
        qpcr_positive=("qpcr_detected", lambda s: (s == "detected").sum()),
        seq_detected=("sequencing_detected", lambda s: (s == "detected").sum()),
    )
    monthly["qpcr_rate"] = monthly["qpcr_positive"] / monthly["n"]
    monthly["seq_rate"] = monthly["seq_detected"] / monthly["n"]
    monthly = monthly.reset_index()
    monthly["month_ts"] = monthly["month"].dt.to_timestamp()
    print(f"{sheet}: {len(df)}/{before} rows kept (mismatch_flag_after_correction == none), {len(monthly)} months")
    return monthly


def make_timeseries_chart(virus, monthly):
    fig, ax_ts = plt.subplots(figsize=(FIG_WIDTH_IN, 6))

    ax_ts.plot(monthly["month_ts"], monthly["qpcr_rate"] * 100, color=COLOR_QPCR,
               marker="o", markersize=4, linewidth=2, label="qPCR positive rate")
    ax_ts.plot(monthly["month_ts"], monthly["seq_rate"] * 100, color=COLOR_SEQ,
               marker="o", markersize=4, linewidth=2, label="Sequencing detected rate")
    ax_ts.set_ylabel("Monthly Positivity Rate", fontsize=18, fontweight="bold")
    ax_ts.set_ylim(0, 105)
    ax_ts.yaxis.set_major_formatter(lambda y, pos: f"{y:.0f}%")
    ax_ts.grid(axis="y", color="#e1e0d9", linewidth=0.8, zorder=0)
    ax_ts.set_axisbelow(True)
    ax_ts.spines["top"].set_visible(False)
    ax_ts.spines["right"].set_visible(False)
    ax_ts.legend(loc="upper left", frameon=True, fontsize=10)
    ax_ts.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    ax_ts.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax_ts.set_xlim(*SHARED_XLIM)
    ax_ts.set_xlabel("Date (Monthly)", fontsize=18, fontweight="bold", labelpad=10)
    for label in ax_ts.get_xticklabels():
        label.set_rotation(30)
        label.set_ha("right")

    ax_ts.set_title(f"{virus['title']}: monthly qPCR vs. sequencing positivity over time",
                     fontsize=15, fontweight="bold", loc="left", pad=15)
    
    fig.subplots_adjust(left=AXES_LEFT, right=AXES_RIGHT, top=0.88, bottom=0.24)

    out_dir = os.path.dirname(virus["out_path_ts"])
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.savefig(virus["out_path_ts"], dpi=300)
    fig.savefig(virus["out_path_ts"].replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved {virus['out_path_ts']}")


def make_scatter_chart(virus, monthly, pearson_r, pearson_p, spearman_r, spearman_p):
    fig, ax_scatter = plt.subplots(figsize=(8, 7))

    ax_scatter.scatter(monthly["qpcr_rate"] * 100, monthly["seq_rate"] * 100,
                        color=COLOR_QPCR, alpha=0.6, s=40, zorder=3)
    # Diagonal drawn over the "real" 0-100 range; axis limits pulled a few
    # points past 0 on each side so points sitting at/near 0% aren't flush
    # against the spines -- reads as "raised" off the corner.
    ax_scatter.plot([0, 100], [0, 100], color="#999999", linestyle=(0, (4, 3)), linewidth=1, zorder=2)
    ax_scatter.set_xlim(-5, 105)
    ax_scatter.set_ylim(-5, 105)
    ax_scatter.xaxis.set_major_formatter(lambda y, pos: f"{y:.0f}%")
    ax_scatter.yaxis.set_major_formatter(lambda y, pos: f"{y:.0f}%")
    ax_scatter.set_xlabel("qPCR positive rate", fontsize=18, fontweight="bold")
    ax_scatter.set_ylabel("Sequencing detected rate", fontsize=18, fontweight="bold")
    ax_scatter.grid(color="#e1e0d9", linewidth=0.8, zorder=0)
    ax_scatter.set_axisbelow(True)
    ax_scatter.spines["top"].set_visible(False)
    ax_scatter.spines["right"].set_visible(False)
    ax_scatter.text(
        0.03, 0.97,
        #f"Pearson r = {pearson_r:.2f} (p = {pearson_p:.1e})\n"
        f"Spearman ρ = {spearman_r:.2f} (p = {spearman_p:.1e})\n"
        f"n = {len(monthly)} months",
        transform=ax_scatter.transAxes, fontsize=16, va="top", ha="left",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="#cccccc", alpha=0.9),
    )
    ax_scatter.set_title(f"{virus['title']}: monthly rate correlation", fontsize=14, fontweight="bold", loc="left", pad=15)

    fig.tight_layout()

    out_dir = os.path.dirname(virus["out_path_scatter"])
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    fig.savefig(virus["out_path_scatter"], dpi=300, bbox_inches="tight", pad_inches=0.3)
    fig.savefig(virus["out_path_scatter"].replace(".png", ".svg"), dpi=300, bbox_inches="tight", pad_inches=0.3)
    plt.close(fig)
    print(f"Saved {virus['out_path_scatter']}")


def make_chart(virus):
    monthly = load_monthly_rates(virus["sheet"])

    pearson_r, pearson_p = pearsonr(monthly["qpcr_rate"], monthly["seq_rate"])
    spearman_r, spearman_p = spearmanr(monthly["qpcr_rate"], monthly["seq_rate"])
    print(f"{virus['title']}: Pearson r={pearson_r:.3f} (p={pearson_p:.2e}), "
          f"Spearman rho={spearman_r:.3f} (p={spearman_p:.2e}) (n={len(monthly)} months)")

    make_timeseries_chart(virus, monthly)
    make_scatter_chart(virus, monthly, pearson_r, pearson_p, spearman_r, spearman_p)


def main():
    for virus in VIRUSES:
        make_chart(virus)


if __name__ == "__main__":
    main()


fluA: 3910/3910 rows kept (mismatch_flag_after_correction == none), 44 months
Influenza A: Pearson r=0.639 (p=3.07e-06), Spearman rho=0.691 (p=2.07e-07) (n=44 months)
Saved monthly_qpcr_vs_sequencing_positivity_fluA_timeseries.png
Saved monthly_qpcr_vs_sequencing_positivity_fluA_correlation.png
fluB: 3897/3897 rows kept (mismatch_flag_after_correction == none), 44 months
Influenza B: Pearson r=0.531 (p=2.09e-04), Spearman rho=0.582 (p=3.46e-05) (n=44 months)
Saved monthly_qpcr_vs_sequencing_positivity_fluB_timeseries.png
Saved monthly_qpcr_vs_sequencing_positivity_fluB_correlation.png
rsvA: 3879/3879 rows kept (mismatch_flag_after_correction == none), 44 months
RSV A: Pearson r=0.652 (p=1.63e-06), Spearman rho=0.526 (p=2.47e-04) (n=44 months)
Saved monthly_qpcr_vs_sequencing_positivity_rsvA_timeseries.png
Saved monthly_qpcr_vs_sequencing_positivity_rsvA_correlation.png
rsvB: 3914/3914 rows kept (mismatch_flag_after_correction == none), 44 months
RSV B: Pearson r=0.696 (p=1.59e-07), Spe